In [7]:
import os
import glob
import random
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image

# Dice coefficient function
def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)      # Convert logits to probabilities
    pred = (pred > 0.5).float()       # Binarize predictions
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.item()

# Define DoubleConv and UNet architecture as used in training
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        self.conv_down1 = DoubleConv(in_channels, 64)
        self.conv_down2 = DoubleConv(64, 128)
        self.conv_down3 = DoubleConv(128, 256)
        self.conv_down4 = DoubleConv(256, 512)
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = DoubleConv(512, 1024)
        
        self.uptrans1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)
        self.uptrans2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)
        self.uptrans3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)
        self.uptrans4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        self.output = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.conv_down1(x)
        x2 = self.maxpool(x1)
        x2 = self.conv_down2(x2)
        x3 = self.maxpool(x2)
        x3 = self.conv_down3(x3)
        x4 = self.maxpool(x3)
        x4 = self.conv_down4(x4)
        x5 = self.maxpool(x4)
        # Bottleneck
        x5 = self.bottleneck(x5)
        # Decoder
        x6 = self.uptrans1(x5)
        x6 = torch.cat([x4, x6], dim=1)
        x6 = self.conv_up1(x6)
        x7 = self.uptrans2(x6)
        x7 = torch.cat([x3, x7], dim=1)
        x7 = self.conv_up2(x7)
        x8 = self.uptrans3(x7)
        x8 = torch.cat([x2, x8], dim=1)
        x8 = self.conv_up3(x8)
        x9 = self.uptrans4(x8)
        x9 = torch.cat([x1, x9], dim=1)
        x9 = self.conv_up4(x9)
        out = self.output(x9)
        return out

# Set device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load trained model (using state_dict)
model = UNet(in_channels=3, out_channels=1).to(device)
model_path = "unet_cornea_segmentation_no_earlystop.pth"
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

# Define test transform (resize to 256x256 and convert to tensor)
test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# Directories for images and labels
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

# Get list of image files with .jpg extension
image_files = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))
if len(image_files) < 20:
    raise ValueError("Not enough images available. Need at least 20 images.")
selected_images = random.sample(image_files, 50)

# Process and evaluate 20 randomly selected images
for img_path in selected_images:
    # Determine the corresponding mask filename (mask extension is .png)
    img_filename = os.path.basename(img_path)  # e.g., "100.jpg"
    base_name = os.path.splitext(img_filename)[0]  # "100"
    mask_filename = base_name + ".png"
    mask_path = os.path.join(labels_dir, mask_filename)

    if not os.path.exists(mask_path):
        print(f"Mask not found for image {img_filename}. Skipping this image.")
        continue

    # Load image and mask using PIL
    image = Image.open(img_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")

    # Apply test transformation
    image_tensor = test_transform(image)
    mask_tensor = test_transform(mask)
    mask_tensor = (mask_tensor > 0.5).float()  # Binarize the mask

    # Add batch dimension and move to device
    input_tensor = image_tensor.unsqueeze(0).to(device)

    # Model inference
    with torch.no_grad():
        output = model(input_tensor)

    # Threshold output after sigmoid
    pred_mask = torch.sigmoid(output)
    pred_mask = (pred_mask > 0.5).float()

    # Calculate Dice coefficient
    dice_score = dice_coefficient(output, mask_tensor.unsqueeze(0).to(device))

    # Convert tensors to numpy arrays for visualization
    image_np = np.array(image.resize((256, 256)))
    mask_np = mask_tensor.squeeze().cpu().numpy()
    pred_mask_np = pred_mask.squeeze().cpu().numpy()

    # Display the original image, ground truth mask, and predicted mask with Dice score
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='white')
    
    axes[0].imshow(image_np)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    
    axes[1].imshow(mask_np, cmap='gray')
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")
    
    axes[2].imshow(pred_mask_np, cmap='gray')    
    axes[2].set_title(f"Predicted Mask\nDice: {dice_score:.4f}")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()


In [8]:
import os
import glob
import torch
import numpy as np
from torchvision import transforms
from PIL import Image
import torch.nn as nn

# Define segmentation metrics
def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.item()

def iou(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection + epsilon
    return (intersection / union).item()

def precision(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    tp = (pred * target).sum()  # True Positives
    fp = (pred * (1 - target)).sum()  # False Positives
    return (tp / (tp + fp + epsilon)).item()

def recall(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    tp = (pred * target).sum()  # True Positives
    fn = ((1 - pred) * target).sum()  # False Negatives
    return (tp / (tp + fn + epsilon)).item()

def f1_score(prec, rec, epsilon=1e-6):
    return (2 * prec * rec + epsilon) / (prec + rec + epsilon)

# Define UNet model (same as training)
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        self.conv_down1 = DoubleConv(in_channels, 64)
        self.conv_down2 = DoubleConv(64, 128)
        self.conv_down3 = DoubleConv(128, 256)
        self.conv_down4 = DoubleConv(256, 512)
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = DoubleConv(512, 1024)
        
        self.uptrans1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)
        self.uptrans2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)
        self.uptrans3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)
        self.uptrans4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        self.output = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.conv_down1(x)
        x2 = self.maxpool(x1)
        x2 = self.conv_down2(x2)
        x3 = self.maxpool(x2)
        x3 = self.conv_down3(x3)
        x4 = self.maxpool(x3)
        x4 = self.conv_down4(x4)
        x5 = self.maxpool(x4)

        x5 = self.bottleneck(x5)

        x6 = self.uptrans1(x5)
        x6 = torch.cat([x4, x6], dim=1)
        x6 = self.conv_up1(x6)

        x7 = self.uptrans2(x6)
        x7 = torch.cat([x3, x7], dim=1)
        x7 = self.conv_up2(x7)

        x8 = self.uptrans3(x7)
        x8 = torch.cat([x2, x8], dim=1)
        x8 = self.conv_up3(x8)

        x9 = self.uptrans4(x8)
        x9 = torch.cat([x1, x9], dim=1)
        x9 = self.conv_up4(x9)

        out = self.output(x9)
        return out

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load trained model
model = UNet(in_channels=3, out_channels=1).to(device)
model_path = "unet_cornea_segmentation2.pth"
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

# Define test transform
test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# Directories for images and labels
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

# Get all image files
image_files = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))
num_images = len(image_files)

# Initialize metric accumulators
dice_scores = []
iou_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# Process all images
for img_path in image_files:
    img_filename = os.path.basename(img_path)
    base_name = os.path.splitext(img_filename)[0]
    mask_filename = base_name + ".png"
    mask_path = os.path.join(labels_dir, mask_filename)

    if not os.path.exists(mask_path):
        continue  # Skip if mask not found

    # Load image and mask
    image = Image.open(img_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")

    # Apply transformations
    image_tensor = test_transform(image)
    mask_tensor = test_transform(mask)
    mask_tensor = (mask_tensor > 0.5).float()

    # Add batch dimension and move to device
    input_tensor = image_tensor.unsqueeze(0).to(device)

    # Model inference
    with torch.no_grad():
        output = model(input_tensor)

    # Compute metrics
    dice = dice_coefficient(output, mask_tensor.unsqueeze(0).to(device))
    iou_val = iou(output, mask_tensor.unsqueeze(0).to(device))
    prec = precision(output, mask_tensor.unsqueeze(0).to(device))
    rec = recall(output, mask_tensor.unsqueeze(0).to(device))
    f1 = f1_score(prec, rec)

    # Store results
    dice_scores.append(dice)
    iou_scores.append(iou_val)
    precision_scores.append(prec)
    recall_scores.append(rec)
    f1_scores.append(f1)

# Compute mean metrics
mean_dice = np.mean(dice_scores)
mean_iou = np.mean(iou_scores)
mean_precision = np.mean(precision_scores)
mean_recall = np.mean(recall_scores)
mean_f1 = np.mean(f1_scores)

# Display results
metrics = {
    "Mean Dice Coefficient": mean_dice,
    "Mean IoU": mean_iou,
    "Mean Precision": mean_precision,
    "Mean Recall": mean_recall,
    "Mean F1-Score": mean_f1
}
metrics
